# OpenMath → Indonesian (v2 — math-safe segmentation)

**Why v1 failed** (`opus-mt-en-id` + placeholder masking):
- Weak model mangled underscore placeholders (`__M0__` → `__M1_`) → LaTeX never restored.
- Pure-LaTeX answer became one placeholder, model dropped it → `jawaban` empty.
- Subword tokenizer split `_` boundaries → `\binom{n_i}}@a_i}` garbage.

**v2 approach — the model never sees a single math symbol:**
1. Split each field into NL spans vs math spans (`\(...\)`, `\[...\]`, `$...$`, `$$...$$`, stray `\boxed{}`).
2. Translate **only** the natural-language spans (NLLB-200, strong en→id).
3. Splice the original math back **verbatim**. ⇒ zero variable/symbol corruption, zero dropped LaTeX.
4. `expected_answer` is usually pure LaTeX → 0 NL spans → copied verbatim into `jawaban` (never empty again).

Batches NL spans across many entries for GPU throughput. Checkpoints per block. Resumable.

**Install (your torch env):**
```
pip install transformers sentencepiece sacremoses accelerate tqdm
```
(torch already present.)

In [18]:
# ── CONFIG ──────────────────────────────────────────────────────
INPUT_PATH  = "hugging_face_AIMO/openmath_reasoning_20k.json"
OUTPUT_DIR  = "hugging_face_AIMO/parts_v2"

START       = 5000    # inclusive  (continue right after the 0:5000 chunk)
END         = 20000   # exclusive  (chunk 2: entries 5000..19999)

MODEL_NAME  = "facebook/nllb-200-distilled-1.3B"  # strong en->id; ~2.6 GB fp16, fits 4060 8 GB
# fallback if VRAM/quality tradeoff needed: "facebook/nllb-200-distilled-600M"
SRC_LANG    = "eng_Latn"
TGT_LANG    = "ind_Latn"

RUN_CHUNK   = 500     # entries to process THIS run, then stop. Re-run the main-loop
                      # cell to continue from the last checkpoint (clears VRAM each run).
BLOCK_SIZE  = 100     # entries processed + checkpointed per block
BATCH_SIZE  = 8       # NL spans per GPU forward pass; auto-halves on OOM
NUM_BEAMS   = 2       # 1 = fastest, 4 = best; 2 = balanced
MAX_SRC_TOK = 256
MAX_TGT_TOK = 400
MAX_SPAN_CHARS = 600  # NL spans longer than this get sentence-split before translation
USE_FP16    = True
# ─────────────────────────────────────────────────────────

In [19]:
import json, re, gc
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: no CUDA — running on CPU (slow)")

OUT_DIR = Path(OUTPUT_DIR); OUT_DIR.mkdir(parents=True, exist_ok=True)
PART_FILE = OUT_DIR / f"part_{START:05d}_{END:05d}.jsonl"
print("Output:", PART_FILE)

GPU : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.2 GB
Output: hugging_face_AIMO/parts_v2/part_05000_20000.jsonl


In [20]:
# ── Segmentation: split text into NL vs math, never feed math to the model ──

# Order matters: $$ before $, display before inline.
MATH_PATTERN = re.compile(
    r"(\$\$.*?\$\$"                              # $$ ... $$
    r"|\\\[.*?\\\]"                              # \[ ... \]
    r"|\\\(.*?\\\)"                              # \( ... \)
    r"|\$[^$\n]*?\$"                             # $ ... $
    r"|\\boxed\{(?:[^{}]|\{[^{}]*\})*\}"        # stray \boxed{...}
    r")",
    re.DOTALL,
)

SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")


def strip_think(text: str) -> str:
    idx = text.find("</think>")
    return text[idx + 8:].strip() if idx >= 0 else text.strip()


def has_alpha(s: str) -> bool:
    return any(c.isalpha() for c in s)


def _sentence_chunks(s: str, max_chars: int):
    """Split a long NL span into <=max_chars pieces on sentence boundaries."""
    if len(s) <= max_chars:
        return [s]
    out, cur = [], ""
    for sent in SENT_SPLIT.split(s):
        cand = (cur + " " + sent).strip() if cur else sent
        if len(cand) > max_chars and cur:
            out.append(cur); cur = sent
        else:
            cur = cand
    if cur:
        out.append(cur)
    return out or [s]


def build_template(text: str):
    """Return list of [kind, content] segments. kind in {'math','txt'}.
    Long 'txt' segments are pre-split so each translatable piece fits the model."""
    parts = MATH_PATTERN.split(text)
    segs = []
    for i, p in enumerate(parts):
        if i % 2 == 1:            # captured math
            segs.append(["math", p])
        elif p:                   # plain text
            if len(p) > MAX_SPAN_CHARS:
                for piece in _sentence_chunks(p, MAX_SPAN_CHARS):
                    segs.append(["txt", piece])
            else:
                segs.append(["txt", p])
    return segs


# quick self-test
_demo = r"The square root of \(z = a+bi\) is \[\boxed{\sqrt{r}\,e^{i\theta/2}}\] which is real."
for k, c in build_template(_demo):
    print(f"  {k:4} | {c}")

  txt  | The square root of 
  math | \(z = a+bi\)
  txt  |  is 
  math | \[\boxed{\sqrt{r}\,e^{i\theta/2}}\]
  txt  |  which is real.


In [21]:
print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG
dtype = torch.float16 if (USE_FP16 and device.type == "cuda") else torch.float32
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
model.eval()

# resolve forced BOS for target language (handles both old/new tokenizer APIs)
TGT_ID = tokenizer.convert_tokens_to_ids(TGT_LANG)
if TGT_ID is None or TGT_ID == tokenizer.unk_token_id:
    TGT_ID = tokenizer.lang_code_to_id[TGT_LANG]
print("forced_bos_token_id:", TGT_ID)
if device.type == "cuda":
    print(f"Model VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading facebook/nllb-200-distilled-1.3B ...
forced_bos_token_id: 256075
Model VRAM: 5.70 GB


In [22]:
@torch.no_grad()
def translate_batch(texts):
    """Translate a list of NL strings en->id. Math never reaches here."""
    if not texts:
        return []
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_SRC_TOK).to(device)
    gen = model.generate(**enc, forced_bos_token_id=TGT_ID,
                         max_length=MAX_TGT_TOK, num_beams=NUM_BEAMS)
    return tokenizer.batch_decode(gen, skip_special_tokens=True)


def reattach_ws(original: str, translated: str) -> str:
    """Model strips edge whitespace; restore it so splice with math stays clean."""
    lead = original[: len(original) - len(original.lstrip())]
    trail = original[len(original.rstrip()):]
    return lead + translated.strip() + trail

In [23]:
print(f"Reading {INPUT_PATH} (slice {START}:{END}) ...")
with open(INPUT_PATH, encoding="utf-8") as f:
    all_data = json.load(f)
data = all_data[START:END]
del all_data; gc.collect()
print(f"Slice size: {len(data):,} entries")

# source field -> output key
FIELD_MAP = [("problem", "soal"),
             ("generated_solution", "cara"),
             ("expected_answer", "jawaban")]
OUT_KEYS = [k for _, k in FIELD_MAP]

# resume from checkpoint
done = 0
if PART_FILE.exists():
    with open(PART_FILE, encoding="utf-8") as f:
        done = sum(1 for line in f if line.strip())
    print(f"Resuming: {done:,} records already on disk, skipping them")
else:
    print("No checkpoint, starting fresh")

Reading hugging_face_AIMO/openmath_reasoning_20k.json (slice 5000:20000) ...
Slice size: 15,000 entries
Resuming: 8,000 records already on disk, skipping them


In [24]:
def process_block(entries):
    """Build templates for a block, batch-translate all unique NL spans, reassemble."""
    templates = []                 # per entry: {out_key: [segments]}
    uniq = {}                      # nl string -> index in pool (dedup)
    pool = []                      # unique strings to translate
    refs = []                      # (entry_i, out_key, seg_i, pool_i)

    for ei, entry in enumerate(entries):
        tmpl = {}
        for field, out_key in FIELD_MAP:
            raw = entry.get(field, "") or ""
            if field == "generated_solution":
                raw = strip_think(raw)
            raw = raw.strip()
            segs = build_template(raw)
            for si, (kind, content) in enumerate(segs):
                if kind == "txt" and has_alpha(content):
                    key = content
                    if key not in uniq:
                        uniq[key] = len(pool); pool.append(content)
                    refs.append((ei, out_key, si, uniq[key]))
            tmpl[out_key] = segs
        templates.append(tmpl)

    # translate unique pool in batches
    out = [None] * len(pool)
    for i in range(0, len(pool), BATCH_SIZE):
        out[i:i + BATCH_SIZE] = translate_batch(pool[i:i + BATCH_SIZE])

    # write translations back (preserving edge whitespace)
    for ei, out_key, si, pi in refs:
        seg = templates[ei][out_key][si]
        seg[1] = reattach_ws(seg[1], out[pi])

    # reassemble — math spliced back verbatim
    records = []
    for tmpl in templates:
        records.append({k: "".join(s[1] for s in tmpl[k]) for k in OUT_KEYS})
    return records


def flush(records):
    with open(PART_FILE, "a", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [ ]:
# ── Main loop: block by block, checkpoint after each block ──
total = len(data)
pbar = tqdm(total=total, initial=done, desc=f"Translating [{START}:{END}]")

for lo in range(done, total, BLOCK_SIZE):
    hi = min(lo + BLOCK_SIZE, total)
    records = process_block(data[lo:hi])
    flush(records)
    pbar.update(hi - lo)
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
pbar.close()
print(f"\nDone. Records in {PART_FILE.name}: {sum(1 for l in open(PART_FILE) if l.strip()):,}")

Translating [5000:20000]:  53%|#####3    | 8000/15000 [00:00<?, ?it/s]

In [ ]:
# ── Sanity check: verify math preserved + jawaban non-empty ──
with open(PART_FILE, encoding="utf-8") as f:
    lines = [l for l in f if l.strip()]
print(f"Total lines: {len(lines):,}")

empty_jawaban = sum(1 for l in lines if not json.loads(l)["jawaban"].strip())
print(f"Empty jawaban: {empty_jawaban} / {len(lines)}")

for label, line in [("FIRST", lines[0]), ("LAST", lines[-1])]:
    r = json.loads(line)
    print(f"\n── {label} ──")
    print("soal   :", r["soal"][:300])
    print("cara   :", r["cara"][:300])
    print("jawaban:", r["jawaban"])

Total lines: 5,000
Empty jawaban: 3 / 5000

── FIRST ──
soal   : Mengingat sekelompok \( N \) bola yang terdiri dari \( C \) warna, di mana jumlah bola dalam setiap warna diwakili sebagai \( n_1, n_2, \ldots, n_C \) (dengan \( n_1 + n_2 + \ldots + n_C = N \)), berapa probabilitas bahwa ketika \( A \) bola dipilih secara acak (di mana \( A \leq N \)), bola yang di
cara   : Untuk menemukan probabilitas bahwa ketika \( A \) bola dipilih secara acak dari sekelompok \( N \) bola yang terdiri dari \( C \) warna, di mana jumlah bola dalam setiap warna diwakili sebagai \( n_1, n_2, \ldots, n_C \) (dengan \( n_1 + n_2 + \ldots + n_C = N \)), bola yang dipilih terdiri dari \( 
jawaban: \(\frac{C_{n_1}^{a_1} \cdot C_{n_2}^{a_2} \cdot \ldots \cdot C_{n_C}^{a_C}}{C_N^A}\)

── LAST ──
soal   : Berapa akar kuadrat dari bilangan kompleks \(a + bi\) dalam hal \(a\) Dan \(b\)?
cara   : Untuk mencari akar kuadrat dari bilangan kompleks \(a + bi\) dalam hal \(a\) Dan \(b\), kita dapat mengikuti langkah-la

In [ ]:
# ── OPTIONAL: convert the JSONL part to a single JSON array ──
# OUT_JSON = OUT_DIR / f"openmath_indo_{START:05d}_{END:05d}.json"
# with open(PART_FILE, encoding="utf-8") as f:
#     recs = [json.loads(l) for l in f if l.strip()]
# with open(OUT_JSON, "w", encoding="utf-8") as f:
#     json.dump(recs, f, ensure_ascii=False, indent=2)
# print(f"Wrote {len(recs):,} records -> {OUT_JSON}")